In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install -q transformers datasets evaluate accelerate sentencepiece sacrebleu rouge-score bert-score scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 88.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cud

In [3]:
import transformers, datasets, sacrebleu, bert_score, rouge_score, sklearn
print("training/eval imports ok")

training/eval imports ok


In [4]:
!pip install -q sentence-transformers

In [5]:
!git clone https://github.com/ffaustin17/role-aware-crisis-summarization.git
%cd role-aware-crisis-summarization
!git status --short --branch 

Cloning into 'role-aware-crisis-summarization'...
remote: Enumerating objects: 211, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 211 (delta 101), reused 163 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (211/211), 7.81 MiB | 17.13 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/kaggle/working/role-aware-crisis-summarization
## main...origin/main


In [6]:
!pip install "minicheck @ git+https://github.com/Liyan06/MiniCheck.git@main"

  Cloning https://github.com/Liyan06/MiniCheck.git (to revision main) to /tmp/pip-install-mpr6351g/minicheck_79122208efd54a27b5ff99c522650746
  Running command git clone --filter=blob:none --quiet https://github.com/Liyan06/MiniCheck.git /tmp/pip-install-mpr6351g/minicheck_79122208efd54a27b5ff99c522650746
  Resolved https://github.com/Liyan06/MiniCheck.git to commit b58b9fa69acbd1015ec970fa65dd752413a053d2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for minicheck: filename=minicheck-0.1.0-py3-none-any.whl size=17615 sha256=b0ffc1c467ccf6ec5c8c4c20c8259d505f99b243e24448d4ae6d6e69bc94d18f
  Stored in directory: /tmp/pip-ephem-wheel-cache-qb11aq9e/wheels/35/b4/0f/98b1775c6b3d3577579dcf13d856c2720a89a99a36d20a426d
Successfully built minicheck


In [7]:
!python scripts/score_rewards.py \
--input data/modeling/t5_baseline_v1/test_predictions.jsonl \
--factuality-backend minicheck \
--output data/rewards/t5_baseline_v1_reward_scores_tweet_relevance_minicheck.jsonl \
--summary-csv reports/tables/t5_baseline_reward_summary_tweet_relevance_minicheck.csv \
--summary-md reports/tables/t5_baseline_reward_summary_tweet_relevance_minicheck.md

config_sentence_transformers.json: 100%|████████| 116/116 [00:00<00:00, 605kB/s]
README.md: 10.5kB [00:00, 32.8MB/s]
sentence_bert_config.json: 100%|██████████████| 53.0/53.0 [00:00<00:00, 371kB/s]
config.json: 100%|█████████████████████████████| 612/612 [00:00<00:00, 3.96MB/s]
model.safetensors: 100%|████████████████████| 90.9M/90.9M [00:00<00:00, 108MB/s]
Loading weights: 100%|█| 103/103 [00:00<00:00, 1521.84it/s, Materializing param=
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 100%|███████████████████| 350/350 [00:00<00:00, 2.28MB/s]
vocab.txt: 232kB [00:00, 7.50MB/s]
tokenizer.json: 466kB [00:00, 22.6MB/s]
config.json: 100%|█████████████████████████████| 826/826 [00:00<00:00, 5.26MB/s]
pytor